# 1. The Coroutine Object
```text
When you define a standard function using def, calling it executes the code block immediately from top to bottom.

When you define a function using async def, calling it does not execute the code. Instead, it returns a coroutine object.

A coroutine is essentially a state machine. Because it is a native Python object, it hooks directly into the Python data model by implementing specific dunder methods—most importantly, __await__. This method allows the object to yield its internal state and control flow back to the caller without destroying its local variables.

In [4]:
import asyncio

async def fetch_data():
    print("Fetching data from DB...")
    await asyncio.sleep(2) # Simulating an I/O pause
    print("Data fetched!")
    return {"id": 1}

# Calling the function DOES NOT run the print statements.
# It instantiates a coroutine object.
coro_object = fetch_data()

print(type(coro_object)) 
# Output: <class 'coroutine'>

print(hasattr(coro_object, '__await__')) 
# Output: True

<class 'coroutine'>
True


C:\Users\shubh\AppData\Local\Temp\ipykernel_24560\283655613.py:11: RuntimeWarning: coroutine 'fetch_data' was never awaited
  coro_object = fetch_data()


* The fetch_data function acts as a factory that creates a coroutine object. 
* That object holds the state of the function— it knows what line of code it is currently on and what local variables exist. 
* But the object cannot execute itself. It needs a driver.

# 2. The Event Loop (The Driver)
```text
The event loop is the engine that drives coroutine objects forward. At its core, an event loop is just an *infinite while True: loop running on a single thread*

It maintains a queue of tasks (wrapped coroutine objects). On each iteration, the loop pulls a task from the queue and executes it until it hits an await keyword attached to an I/O operation (like waiting for our raw TCP socket to receive bytes).

```mermaid
flowchart LR

    MT["Main Thread"]

    subgraph EL["Event Loop"]
        direction TB
        TQ["Task Queue"]
        MQ(("Monitor<br/>Task Queue"))
        RT["Run Tasks until<br/>blocking I/O,<br/>pause them,<br/>hand over to OS"]
        CI["Check for<br/>completed I/O<br/>& unpause the tasks"]

        TQ --> MQ
        MQ --> RT
        RT --> CI
        CI --> MQ
    end

    subgraph OS["OS"]
        direction TB
        IO["Handle I/O tasks"]
        DONE["Notify the<br/>I/O Task is done"]

        IO --> DONE
    end

    MT -->|Submit Tasks| TQ
    RT -->|Blocking I/O| IO
    DONE -->|I/O completed| CI
```

- **Main Thread**
  - The main Python program starts here.
  - It submits asynchronous tasks to the event loop.

- **Task Queue**
  - Holds tasks that are ready to run.
  - The event loop picks tasks from this queue.

- **Monitor Task Queue**
  - The event loop continuously checks for tasks that are ready to execute.
  - It decides which task should run next.

- **Run Task**
  - The event loop starts or resumes a task.
  - The task runs until it reaches an `await` that requires waiting.

- **Task reaches I/O**
  - Example: waiting for a network response, database response, or socket data.
  - The task cannot continue until the I/O operation is ready.
  - The event loop pauses that task.

- **Hand over I/O to OS**
  - The operating system handles the underlying I/O operation.
  - The event loop doesn't continuously sit there waiting for it.

- **Event Loop runs other tasks**
  - While one task is waiting for I/O, the event loop can run other ready tasks.
  - This is the main benefit of asynchronous programming.

- **OS completes I/O**
  - The OS detects that the I/O operation has completed.
  - It notifies the event-loop system that the operation is ready.

- **Check completed I/O**
  - The event loop checks which paused tasks can continue.
  - It finds the task that was waiting for that I/O.

- **Unpause/Resume Task**
  - The event loop resumes the task from where it stopped at `await`.
  - The task continues executing.

- **Repeat**
  - The event loop keeps repeating this process:
    - Find ready task
    - Run task
    - Pause when it reaches `await`
    - Let OS handle I/O
    - Check for completed I/O
    - Resume tasks
    - Repeat

- **Core Idea**
  - `await` means: **"I need to wait, so pause me and let other tasks run."**
  - This allows a single thread to efficiently handle many I/O-bound tasks.

### The Handoff Mechanism
```text
When a task hits await network_call(), two things happen:

* The coroutine object saves its exact state (variables, current line of execution) into memory.

* It yields control back to the event loop.

The event loop says, "Okay, you are blocked waiting for the network. I'm going to put you aside and run the next task in the queue."

This is cooperative multitasking. The tasks must voluntarily give up control (await) so the loop can move on. If a task executes a heavy for loop computing prime numbers, it never yields control, and the entire event loop freezes.

# 3. How the Loop Knows When to Resume
If the event loop puts a paused task aside, how does it know when the TCP socket has finally received data so it can resume the task?

It relies on the Operating System. Python's asyncio uses the selectors module, which taps directly into highly efficient OS-level event notification systems (like epoll on Linux or kqueue on macOS).